In [6]:
import pandas as pd
from src.lista4.data import cargar_datos
from src.lista4.models import ajustar_regresion_lineal
from src.lista4.metrics import riesgo_empirico_cuadratico

Iteración sobre archivos y recolección

In [7]:
archivos = ['experimento_2_a.csv', 'experimento_2_b.csv', 'experimento_2_c.csv']
resultados = []

In [8]:
for archivo in archivos:
    df = cargar_datos(f'../data/{archivo}')
    X1, X_ambas, y = df[['x1']], df[['x1', 'x2']], df['y']
    
    mod_x1 = ajustar_regresion_lineal(X1, y)
    mod_ambas = ajustar_regresion_lineal(X_ambas, y)
    
    riesgo_x1 = riesgo_empirico_cuadratico(y, mod_x1.predict(X1))
    riesgo_ambas = riesgo_empirico_cuadratico(y, mod_ambas.predict(X_ambas))
    
    resultados.append({
        'Archivo': archivo,
        'corr(x1,x2)': df['x1'].corr(df['x2']),
        'R_x1': riesgo_x1,
        'R_x1_x2': riesgo_ambas,
        'beta_1': mod_ambas.coef_[0],
        'beta_2': mod_ambas.coef_[1],
        'Delta R': riesgo_x1 - riesgo_ambas
    })

In [9]:
tabla_ej2 = pd.DataFrame(resultados)
display(tabla_ej2)

,Archivo,"corr(x1,x2)",R_x1,R_x1_x2,beta_1,beta_2,Delta R
0,experimento_2_a.csv,0.999760,1.029350,1.025907,5.957231,-2.900650,0.003443
1,experimento_2_b.csv,0.999779,1.061539,1.059304,0.606315,2.325776,0.002235
2,experimento_2_c.csv,0.999821,1.141281,1.140041,4.903450,-1.911232,0.001240


Calcular rangos (max-min)

In [10]:
def calcular_rango(serie):
    return serie.max() - serie.min()

In [11]:
print(f"Rango de beta_1: {calcular_rango(tabla_ej2['beta_1']):.6f}")
print(f"Rango de beta_2: {calcular_rango(tabla_ej2['beta_2']):.6f}")
print(f"Rango de Delta R: {calcular_rango(tabla_ej2['Delta R']):.6f}")

Rango de beta_1: 5.350916
Rango de beta_2: 5.226426
Rango de Delta R: 0.002203


Conclusiones:
   - Cambio en el riesgo empírico: La incorporación de $x_2$ no produce un cambio importante. Al comparar los valores, el $\Delta R$ (la diferencia de riesgo empírico) es minimo en los tres archivos (cercano a 0.003 o menos). Esto significa que agregar $x_2$ casi no reduce el error ni mejora las predicciones en comparación con usar únicamente $x_1$. Además, el rango de $\Delta R$ entre los tres archivos es prácticamente cero, demostrando que este comportamiento es constante.
   
   - Coeficientes: Los coeficientes $\hat{\beta}_1$ y $\hat{\beta}_2$ no cambian en la misma proporción sino, varían drásticamente. Mientras el riesgo se mantiene estable, los coeficientes saltan de un archivo a otro (por ejemplo, pasando de positivos a negativos o cambiando de tamaño abruptamente), lo cual se evidencia al ver que los rangos de $\hat{\beta}_1$ y $\hat{\beta}_2$ superan el valor de 5.

   - Por que? Como $corr(x_1, x_2)$ es casi 0.999 en todos los archivos. Esto es un caso extremo de multicolinealidad. Como las dos variables aportan exactamente la misma información, el modelo se "confunde" tratando de repartir el peso entre ellas, volviendo los coeficientes súper inestables y sensibles a cualquier pequeño cambio en los datos, todo esto sin aportar ninguna mejora real al ajuste (por eso el riesgo empírico ni se mueve).